# Практическое занятие 8. Расчет режима энергосистемы в pandapower

Версия: студенческая.

Цель занятия - выполнить расчет установившегося режима
электроэнергетической системы и интерпретировать напряжения шин,
загрузку линий и изменение режима при росте нагрузки.

## Инициализация среды выполнения

Ячейка ниже обеспечивает запуск блокнота в Google Colab и в
локальном Jupyter Notebook. Если проект уже открыт локально,
повторное клонирование не выполняется.

In [ ]:
# COLAB_BOOTSTRAP_APPailab
from pathlib import Path
import os
import sys

REQUIRED_PROCESSED_FILES = ['practice_08_power_flow_features.csv', 'practice_08_power_flow_diagnostics.csv', 'practice_08_power_flow_scenarios.csv', 'practice_07_09_dataset_catalog.csv', 'practice_07_09_dataset_assignments.csv']
PROJECT_REPOSITORY_URL = "https://github.com/Alexflex/appailab.git"

def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-colab.txt").exists() and (candidate / "data" / "processed").exists():
            return candidate
    return None

project_root = find_project_root(Path.cwd())

if project_root is None:
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB:
        workdir = Path("/content/appailab")
        if not workdir.exists():
            !git clone -q {PROJECT_REPOSITORY_URL} {workdir}
        project_root = workdir
        os.chdir(project_root)
        !pip install -q -r requirements-colab.txt
    else:
        raise FileNotFoundError(
            "Не найден корень проекта. Откройте блокнот из репозитория appailab "
            "или выполните git clone перед запуском."
        )

sys.path.insert(0, str(project_root / "src"))
missing_files = [
    name for name in REQUIRED_PROCESSED_FILES
    if not (project_root / "data" / "processed" / name).exists()
]
if missing_files:
    raise FileNotFoundError(
        "Не найдены подготовленные CSV: " + ", ".join(missing_files)
        + ". Выполните python scripts/generate_datasets.py."
    )

print(f"Корень проекта: {project_root}")
print("Проверенные CSV:", ", ".join(REQUIRED_PROCESSED_FILES))

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

DATA_DIR = project_root / "data" / "processed"
CATALOG_07_09_FILE = DATA_DIR / "practice_07_09_dataset_catalog.csv"
ASSIGNMENTS_07_09_FILE = DATA_DIR / "practice_07_09_dataset_assignments.csv"
RANDOM_STATE = 20260507

## Теоретический блок

Расчет режима энергосистемы (power flow calculation, load flow
calculation) определяет модули и углы напряжений в шинах, потоки
активной и реактивной мощности, а также загрузку линий. AC power
flow - расчет переменного тока, в котором учитываются активная
мощность `P`, реактивная мощность `Q`, модуль напряжения `U` и
фазовый угол.

Баланс мощности в узле:

$$
P_{gen} - P_{load} = P_{injected},\quad
Q_{gen} - Q_{load} = Q_{injected}.
$$

## Последовательность работы

1. Загрузить сценарии нагрузки и генерации.
2. Создать учебную 6-узловую сеть 110 kV в `pandapower`.
3. Выполнить AC power flow для базового сценария.
4. Сравнить базовый режим и режим с увеличенной нагрузкой.
5. Найти слабый узел и наиболее нагруженную линию.
6. Объяснить, почему расчет режима не является заменой машинному
   обучению.

In [ ]:
features_df = pd.read_csv(DATA_DIR / "practice_08_power_flow_features.csv")
diagnostics_df = pd.read_csv(DATA_DIR / "practice_08_power_flow_diagnostics.csv")
scenarios_df = pd.read_csv(DATA_DIR / "practice_08_power_flow_scenarios.csv")

print("features:", features_df.shape)
print("diagnostics:", diagnostics_df.shape)
print("scenarios:", scenarios_df.shape)
display(scenarios_df.head())
display(features_df.head())

## Паспорт сети

Учебная сеть содержит 6 шин напряжением 110 kV, внешнюю сеть,
один генератор и четыре нагрузки. Одна строка сценариев задает
входы расчета. Столбцы с напряжениями и загрузкой линий являются
результатами расчета, а не исходными признаками.

In [ ]:
display(features_df.groupby("scenario_label").agg(
    n=("scenario_id", "count"),
    mean_load=("total_load_mw", "mean"),
    mean_min_vm=("min_vm_pu", "mean"),
    mean_max_loading=("max_line_loading_percent", "mean"),
).round(4))

## Расчет базового режима в pandapower

В этой ячейке используется та же функция построения сети, что и при
генерации CSV. Это позволяет студенту увидеть объект `pandapower`
и результат `runpp`, не переписывая большой код создания сети.

In [ ]:
from appai_lab.data_generators import create_power_flow_network
import pandapower as pp

base_scenario = scenarios_df.iloc[0].to_dict()
net = create_power_flow_network(base_scenario)
pp.runpp(net, algorithm="nr", init="flat", numba=False)
display(net.bus[["name", "vn_kv"]])
display(net.res_bus[["vm_pu", "va_degree"]].round(4))
display(net.res_line[["p_from_mw", "q_from_mvar", "loading_percent"]].round(3))

## Визуальный анализ сценариев

Диаграммы ниже показывают, как рост суммарной нагрузки связан со
снижением минимального напряжения и ростом загрузки наиболее
нагруженной линии.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(features_df["total_load_mw"], features_df["min_vm_pu"], alpha=0.75)
axes[0].set_xlabel("Суммарная нагрузка, MW")
axes[0].set_ylabel("Минимальное напряжение, p.u.")
axes[0].set_title("Нагрузка и слабый узел")

axes[1].scatter(features_df["total_load_mw"], features_df["max_line_loading_percent"], alpha=0.75, color="#c44e52")
axes[1].axhline(100.0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Суммарная нагрузка, MW")
axes[1].set_ylabel("Максимальная загрузка линии, percent")
axes[1].set_title("Нагрузка и критическая линия")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: задайте значение. Задайте множитель нагрузки в диапазоне 1.10..1.30 для сценария чувствительности.
# Рекомендуемое значение: 1.2
load_multiplier = None
if load_multiplier is None:
    raise ValueError('Заполните load_multiplier: Задайте множитель нагрузки в диапазоне 1.10..1.30 для сценария чувствительности.')

In [ ]:
# TODO: задайте значение. Задайте нижний учебный предел напряжения в диапазоне 0.95..0.98 p.u.
# Рекомендуемое значение: 0.97
voltage_limit_pu = None
if voltage_limit_pu is None:
    raise ValueError('Заполните voltage_limit_pu: Задайте нижний учебный предел напряжения в диапазоне 0.95..0.98 p.u.')

## Сравнение базового и измененного режима

Изменение нагрузки применяется к одному расчетному сценарию. Это
не обучение модели, а инженерный численный эксперимент.

In [ ]:
changed = dict(base_scenario)
for bus in [2, 3, 4, 5]:
    changed[f"load_bus_{bus}_mw"] *= load_multiplier
    changed[f"load_bus_{bus}_mvar"] *= load_multiplier

changed_net = create_power_flow_network(changed)
pp.runpp(changed_net, algorithm="nr", init="flat", numba=False)

comparison = pd.DataFrame({
    "bus": net.bus["name"].to_numpy(),
    "base_vm_pu": net.res_bus["vm_pu"].to_numpy(),
    "changed_vm_pu": changed_net.res_bus["vm_pu"].to_numpy(),
})
comparison["delta_vm_pu"] = comparison["changed_vm_pu"] - comparison["base_vm_pu"]
comparison["below_limit_after_change"] = comparison["changed_vm_pu"] < voltage_limit_pu
display(comparison.round(5))

line_comparison = pd.DataFrame({
    "line": net.line["name"].to_numpy(),
    "base_loading_percent": net.res_line["loading_percent"].to_numpy(),
    "changed_loading_percent": changed_net.res_line["loading_percent"].to_numpy(),
})
line_comparison["delta_loading_percent"] = (
    line_comparison["changed_loading_percent"] - line_comparison["base_loading_percent"]
)
display(line_comparison.round(3))

In [ ]:
plt.figure(figsize=(9, 4))
x = np.arange(len(comparison))
plt.plot(x, comparison["base_vm_pu"], marker="o", label="Базовый режим")
plt.plot(x, comparison["changed_vm_pu"], marker="o", label="Измененный режим")
plt.xticks(x, comparison["bus"], rotation=20)
plt.ylabel("Напряжение, p.u.")
plt.title("Напряжения шин до и после роста нагрузки")
plt.legend()
plt.show()

line_comparison.plot(x="line", y=["base_loading_percent", "changed_loading_percent"], kind="bar", figsize=(10, 4))
plt.ylabel("Загрузка линии, percent")
plt.title("Загрузка линий до и после роста нагрузки")
plt.tight_layout()
plt.show()

## АНТИПРИМЕР: расчет режима не является задачей машинного обучения

Нельзя подменять физический расчет простым подбором зависимости без
понимания входов и ограничений сети. Машинное обучение может быть
полезным дополнением, но базовый режим должен проверяться расчетной
моделью и инженерными ограничениями.

In [ ]:
# TODO: впишите краткий вывод. Опишите слабый узел и наиболее нагруженную линию.
power_flow_interpretation = ''
if not power_flow_interpretation.strip():
    raise ValueError('Заполните power_flow_interpretation: Опишите слабый узел и наиболее нагруженную линию.')
print(power_flow_interpretation)

## Источники и проверка актуальности

Базовые занятия 7-9 используют локальные воспроизводимые CSV. Открытые
источники ниже применяются как методические задания: студент должен
определить объект, признаки, целевую переменную, риски утечки данных
и допустимые визуализации.

In [ ]:
catalog_07_09 = pd.read_csv(CATALOG_07_09_FILE)
assignments_07_09 = pd.read_csv(ASSIGNMENTS_07_09_FILE)
display(catalog_07_09[[
    "dataset_id", "name", "lessons", "implementation_status",
    "risk_level", "checked_at"
]])

## Реестр найденных наборов данных и развернутые задания

Все источники имеют статус `methodology_only`, то есть на данном
этапе они не являются обязательными для выполнения в аудитории.

In [ ]:
display(assignments_07_09[[
    "assignment_id", "lesson", "assignment_title",
    "recommended_visualizations", "control_questions"
]])

## Контрольный чек-лист отчета

1. Описаны элементы сети и входные параметры сценария.
2. Приведены таблицы напряжений шин и загрузки линий.
3. Построены графики сравнения базового и измененного режима.
4. Указаны слабый узел и критическая линия.
5. Объяснено отличие расчета режима от модели машинного обучения.